# Evidencias individuales de Capacitación y certificación

Construye una carpeta por tipo en `data/evidencias/`: `capacitaciones/` (`CAPACITACION`) y `certificaciones/` (`CERTIFICACION`).
Reglas en `README.md` y formato de atributos en `../ATRIBUTOS.md`. Requiere haber generado antes las evidencias de trayectoria. Para no duplicar, aplica las reglas de ponencias sobre su fuente.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.capacitacion import construir_evidencias_capacitacion

pd.set_option("display.max_colwidth", 200)


In [2]:
evidencias, reporte, validacion = construir_evidencias_capacitacion(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))


{
  "fecha_corte": "2026-09-29",
  "poblacion_con_trayectoria": 3164,
  "personas_con_evidencias": 2673,
  "evidencias_total": 46081,
  "registros_no_considerados": 2795,
  "subtipos": {
    "CAPACITACION": {
      "fuente": "processed/capacitaciones_todas.csv",
      "filas_poblacion": 46188,
      "filas_no_consideradas_por_motivo": {
        "curso institucional obligatorio sin contenido temático (inducción, ética, valores, PAP)": 2556,
        "ya registrado como certificación": 105,
        "ya registrado como ponencia": 60,
        "nombre sin tema": 15
      },
      "evidencias_con_varias_participaciones": 845,
      "evidencias": 42348,
      "personas": 2566
    },
    "CERTIFICACION": {
      "fuente": "processed/certificados_todos.csv",
      "filas_poblacion": 3979,
      "filas_no_consideradas_por_motivo": {
        "curso institucional obligatorio sin contenido temático (inducción, ética, valores, PAP)": 53,
        "nombre sin tema": 4,
        "registro de título (no e

## Validación (todas las filas deben tener `n_fallas = 0`)


In [3]:
validacion


,chequeo,n_fallas,ok
0,columnas_esperadas,0,True
1,evidencia_id_nulo,0,True
2,evidencia_id_duplicado,0,True
3,persona_fuera_de_poblacion,0,True
4,tipo_id_invalido,0,True
5,atributos_json_invalido,0,True
6,fecha_inicio_nula,0,True
7,fechas_invertidas_sin_marcar,0,True
8,fecha_inicio_futura,0,True
9,texto_vacio,0,True


## Ejemplos por subtipo


In [4]:
for tipo_id, grupo in evidencias.groupby("tipo_id"):
    print("\n=====", tipo_id)
    for _, r in grupo.sample(3, random_state=7).iterrows():
        print(f"- {r.evidencia_id} | {r.texto}")
        print("  ", json.dumps(json.loads(r.atributos), ensure_ascii=False))



===== CAPACITACION
- CAP-CUR-f6330ac4a620 | Introduction to Cybersecurity, Tipo: Curso, País: Estados Unidos, Modalidad: Virtual
   {"nombre": "Introduction to Cybersecurity", "nombres_originales": ["Introduction to Cybersecurity"], "tipos_evento": ["CURSO"], "fecha_inicio": "2020-10-01", "fecha_fin": "2020-10-31", "n_participaciones": 1, "n_registros": 1, "paises": ["ESTADOS UNIDOS"], "es_exterior": true, "modalidades": ["VIRTUAL"], "certificado_por": [], "horas_total": 40.0, "tipos_certificado": [], "codigos_area_capacitacion": [], "codigos_tipo_conocimiento": [], "n_personas_mismo_curso": 14, "participaciones": [{"fecha_inicio": "2020-10-01", "fecha_fin": "2020-10-31", "tipo_evento": "CURSO", "pais": "ESTADOS UNIDOS", "modalidad": "VIRTUAL", "certificado_por": null, "horas": 40.0, "tipo_certificado": null, "codigo_area_capacitacion": null, "codigo_tipo_conocimiento": null, "n_registros": 1, "ids_capacitacion": [44714]}], "_origen": {"fuente": "processed/capacitaciones_todas.csv", "

## Registros no considerados (para revisión)

Conteo por motivo; se listan completos todos salvo los cursos institucionales obligatorios (son miles).


In [5]:
for carpeta in ("capacitaciones", "certificaciones"):
    nc = pd.read_csv(PROJECT_ROOT / "data" / "evidencias" / carpeta / "registros_no_considerados.csv")
    print("=====", carpeta, len(nc))
    print(nc["motivo"].value_counts().to_string())
    print(nc[~nc["motivo"].str.startswith("curso institucional")].to_string(index=False))


===== capacitaciones 2736
motivo
curso institucional obligatorio sin contenido temático (inducción, ética, valores, PAP)    2556
ya registrado como certificación                                                            105
ya registrado como ponencia                                                                  60
nombre sin tema                                                                              15
 persona_id      tipo_id                           motivo  id_origen                                                                                                                                                                                                                                            descripcion fecha_inicio
      10136 CAPACITACION      ya registrado como ponencia      35801                                                                                                                                                                                          